# 10 - Zero-Shot Classification with Cloud LLMs (OpenRouter)

Classifies tweets using large cloud LLMs via the **OpenRouter API** (OpenAI-compatible).

### Models (all free on OpenRouter — no payment required)
| Key | Model | Parameters |
|-----|-------|------------|
| `deepseek` | DeepSeek V4 Flash | 284B MoE (13B active) |
| `qwen` | Qwen3 80B Instruct | 80B (3B active) |
| `llama33` | Llama 3.3 70B | 70B dense |

### Datasets
- **Manchester** - reliable vs misinformation
- **Monkeypox** - reliable vs misinformation
- **PHEME** - not_rumour vs rumour

### Requirements
- `pip install openai`
- `OPENROUTER_API_KEY` environment variable (free key — no credit card — at openrouter.ai)

### Run order
Run once per combination: set `DATASET` + `MODEL`, then run all cells.
9 total runs (3 models × 3 datasets).

## 0. Dataset & Model Selection

In [ ]:
# ============================================================
# CHANGE THESE TWO VARIABLES TO SWITCH RUNS
# DATASET:  'manchester' | 'monkeypox' | 'pheme'
# MODEL:    'deepseek'   | 'qwen'      | 'llama33'
# ============================================================
DATASET = 'manchester'
MODEL   = 'deepseek'

# ── Dataset configs ───────────────────────────────────────────────────────────
DATASET_CONFIG = {
    'manchester': {
        'test':        '../../data/gold_standard/manchester_test.csv',
        'text_col':    'cleaned_tweet',
        'label_col':   'label',
        'label_map':   {'reliable': 0, 'misinformation': 1},
        'label_names': ['reliable', 'misinformation'],
        'pos_label':   'misinformation',
        'topic':       'the 2017 Manchester Arena bombing',
        'class_a':     'reliable',
        'class_b':     'misinformation',
        'class_a_desc': 'factually accurate, verified, or plausible news about the event',
        'class_b_desc': 'false, unverified, or misleading claims — rumours, conspiracy theories, or fabricated stories',
    },
    'monkeypox': {
        'test':        '../../data/gold_standard/monkeypox_test.csv',
        'text_col':    'cleaned_tweet',
        'label_col':   'label',
        'label_map':   {'reliable': 0, 'misinformation': 1},
        'label_names': ['reliable', 'misinformation'],
        'pos_label':   'misinformation',
        'topic':       'the 2022 Monkeypox (Mpox) outbreak',
        'class_a':     'reliable',
        'class_b':     'misinformation',
        'class_a_desc': 'factually accurate health information about Monkeypox symptoms, transmission, or treatment',
        'class_b_desc': 'false health claims, conspiracy theories, or misleading information about Monkeypox',
    },
    'pheme': {
        'test':        '../../data/gold_standard/pheme_test.csv',
        'text_col':    'cleaned_tweet',
        'label_col':   'label',
        'label_map':   {'not_rumour': 0, 'rumour': 1},
        'label_names': ['not_rumour', 'rumour'],
        'pos_label':   'rumour',
        'topic':       'breaking news events (Charlie Hebdo attack 2015, Ferguson unrest 2014)',
        'class_a':     'not_rumour',
        'class_b':     'rumour',
        'class_a_desc': 'verified news, factual reporting, or confirmed information about the events',
        'class_b_desc': 'unverified claims, speculation, or information that has not been confirmed by credible sources',
    },
}

# ── Model configs (free on OpenRouter) ────────────────────────────────────────
# Verify / browse models at: https://openrouter.ai/models?q=free
MODEL_CONFIG = {
    'deepseek': {
        'openrouter_id': 'deepseek/deepseek-v4-flash:free',
        'display_name':  'DeepSeek V4 Flash (284B)',
        'params_b':      284,
    },
    'qwen': {
        'openrouter_id': 'qwen/qwen3-next-80b-a3b-instruct:free',
        'display_name':  'Qwen3 80B Instruct',
        'params_b':      80,
    },
    'llama33': {
        'openrouter_id': 'meta-llama/llama-3.3-70b-instruct:free',
        'display_name':  'Llama 3.3 70B',
        'params_b':      70,
    },
}

CFG       = DATASET_CONFIG[DATASET]
MODEL_CFG = MODEL_CONFIG[MODEL]

print(f"Dataset : {DATASET.upper()}")
print(f"Topic   : {CFG['topic']}")
print(f"Classes : {CFG['label_names']}")
print(f"Model   : {MODEL_CFG['display_name']}")
print(f"OpenRouter ID: {MODEL_CFG['openrouter_id']}")

## 1. Imports & Setup

In [ ]:
import os
import re
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI, APIError, APITimeoutError, RateLimitError

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score,
)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path('../../results')
PREDS_DIR   = RESULTS_DIR / 'predictions'
FIGS_DIR    = RESULTS_DIR / 'figures'
for d in [PREDS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Imports ready.')

## 2. OpenRouter Setup

Set your key before running:
```bash
# Windows PowerShell
$env:OPENROUTER_API_KEY = "sk-or-..."

# or in a .env file loaded below
```

In [ ]:
# Load .env if present (pip install python-dotenv)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_KEY = os.environ.get('OPENROUTER_API_KEY', '')

if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "Set it with: $env:OPENROUTER_API_KEY = 'sk-or-...'"
    )

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=API_KEY,
)

API_TIMEOUT       = 60    # seconds per request
MAX_RETRIES       = 3
RETRY_DELAY_BASE  = 5     # seconds (doubles each retry)
MAX_TOKENS        = 400

print(f'OpenRouter client ready.')
print(f'Model  : {MODEL_CFG["display_name"]}')
print(f'Key    : {API_KEY[:12]}...{API_KEY[-4:]}')

## 3. Load Test Data

In [ ]:
df_test = pd.read_csv(CFG['test'])
df_test.dropna(subset=[CFG['text_col'], CFG['label_col']], inplace=True)
df_test[CFG['text_col']] = df_test[CFG['text_col']].astype(str)

LABEL_MAP = CFG['label_map']
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}

print(f'Test set: {len(df_test):,} samples')
print(f'\nLabel distribution:')
print(df_test[CFG['label_col']].value_counts())

## 4. Promptbook — Dataset-Specific Prompts

In [ ]:
SYSTEM_PROMPT = "You are an expert fact-checker and misinformation analyst specializing in social media content. Always respond with valid JSON only — no extra text before or after."

def build_user_prompt(tweet_text: str, cfg: dict) -> str:
    return f"""Your task: Classify the following tweet about {cfg['topic']}.

CLASSES:
- \"{cfg['class_a']}\": {cfg['class_a_desc']}
- \"{cfg['class_b']}\": {cfg['class_b_desc']}

TWEET:
\"\"\"{tweet_text}\"\"\"

INSTRUCTIONS:
Think step-by-step before classifying. Consider:
1. What specific claim does the tweet make?
2. Does it present verifiable facts, or unverified/emotional claims?
3. Are there signals of misinformation: conspiracy language, extreme emotion, lack of sources, implausible claims?
4. What is your final classification?

Respond in this exact JSON format (no extra text before or after):
{{
  "reasoning": "<your step-by-step reasoning in 2-4 sentences>",
  "label": "{cfg['class_a']}" or "{cfg['class_b']}",
  "confidence": <float between 0.0 and 1.0>
}}"""


# Preview
sample_tweet  = df_test[CFG['text_col']].iloc[0]
sample_prompt = build_user_prompt(sample_tweet, CFG)
print('=== PROMPT PREVIEW ===')
print(sample_prompt)
print(f'\nTweet: {sample_tweet[:200]}')

## 5. Inference with Retry Logic

In [ ]:
def call_api(user_prompt: str, max_retries: int = MAX_RETRIES) -> str:
    """
    Call OpenRouter API. Returns raw response string.
    Retries on timeout and rate-limit errors with exponential backoff.
    Returns empty string on total failure (never raises).
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_CFG['openrouter_id'],
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_prompt},
                ],
                temperature=0.0,
                max_tokens=MAX_TOKENS,
                timeout=API_TIMEOUT,
            )
            raw = response.choices[0].message.content or ''
            return raw

        except RateLimitError:
            wait = RETRY_DELAY_BASE * (2 ** attempt)
            print(f'  [RateLimit] waiting {wait}s before retry {attempt+1}/{max_retries}')
            time.sleep(wait)

        except APITimeoutError:
            print(f'  [Timeout] attempt {attempt+1}/{max_retries}')
            if attempt < max_retries - 1:
                time.sleep(RETRY_DELAY_BASE * (attempt + 1))

        except APIError as e:
            print(f'  [APIError] {e}')
            if attempt < max_retries - 1:
                time.sleep(3)

        except Exception as e:
            print(f'  [Unexpected] {e}')
            if attempt < max_retries - 1:
                time.sleep(3)

    return ''


def parse_response(response_text: str, cfg: dict) -> dict:
    """
    Parse JSON response. Falls back to keyword matching.
    Returns dict with: label, confidence, reasoning, parse_error, parse_method.
    """
    if not response_text or not response_text.strip():
        return {
            'label': None, 'confidence': 0.5,
            'reasoning': 'Empty response from API',
            'parse_error': True, 'parse_method': 'empty',
        }

    # Primary: JSON parse
    try:
        clean = re.sub(r'```json\s*|```\s*', '', response_text).strip()
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            data  = json.loads(match.group())
            label = str(data.get('label', '')).strip().lower()
            if label in cfg['label_map']:
                return {
                    'label':        label,
                    'confidence':   float(data.get('confidence', 0.5)),
                    'reasoning':    str(data.get('reasoning', '')),
                    'parse_error':  False,
                    'parse_method': 'json',
                }
    except (json.JSONDecodeError, ValueError, TypeError):
        pass

    # Fallback: keyword scan
    text_lower = response_text.lower()
    for label_name in sorted(cfg['label_names'], key=len, reverse=True):
        if label_name in text_lower:
            return {
                'label': label_name, 'confidence': 0.5,
                'reasoning': response_text[:300],
                'parse_error': True, 'parse_method': 'keyword_fallback',
            }

    return {
        'label': None, 'confidence': 0.0,
        'reasoning': response_text[:300],
        'parse_error': True, 'parse_method': 'default',
    }


def classify_tweet(tweet: str, cfg: dict) -> dict:
    prompt = build_user_prompt(tweet, cfg)
    raw    = call_api(prompt)
    return parse_response(raw, cfg)


print('Inference functions defined.')
print(f'Model: {MODEL_CFG["display_name"]} | Timeout: {API_TIMEOUT}s | Max retries: {MAX_RETRIES}')

## 6. Run Classification on Test Set

> Cloud API — rate limits apply. Checkpoint every 50 tweets for resume support.

In [ ]:
CHECKPOINT_EVERY = 50

raw_path        = PREDS_DIR / f'{DATASET}_{MODEL}_raw.csv'
checkpoint_path = PREDS_DIR / f'{DATASET}_{MODEL}_checkpoint.csv'

# Resume from checkpoint
done_indices = set()
results      = []

if checkpoint_path.exists():
    df_ckpt      = pd.read_csv(checkpoint_path)
    done_indices = set(df_ckpt['index'].tolist())
    results      = df_ckpt.to_dict('records')
    print(f'Checkpoint found: {len(done_indices):,} tweets already processed — resuming.')
else:
    print('No checkpoint — starting fresh.')

df_todo = df_test[~df_test.index.isin(done_indices)]
total   = len(df_test)

print(f'\nClassifying {len(df_todo):,} remaining tweets with {MODEL_CFG["display_name"]}...')
print(f'(Total: {total:,} | Already done: {len(done_indices):,})\n')

start_time    = time.time()
request_times = []

for n, (i, row) in enumerate(tqdm(df_todo.iterrows(), total=len(df_todo), desc='Classifying'), start=1):
    tweet      = row[CFG['text_col']]
    true_label = row[CFG['label_col']]

    t0     = time.time()
    result = classify_tweet(tweet, CFG)
    t1     = time.time()
    request_times.append(t1 - t0)

    results.append({
        'index':        i,
        'text':         tweet,
        'true_label':   true_label,
        'pred_label':   result['label'],
        'confidence':   result['confidence'],
        'reasoning':    result['reasoning'],
        'parse_error':  result['parse_error'],
        'parse_method': result['parse_method'],
    })

    processed_total = len(done_indices) + n
    avg_time        = sum(request_times) / len(request_times)
    remaining_n     = total - processed_total
    eta_sec         = avg_time * remaining_n

    if n % 10 == 0 or n == len(df_todo):
        print(
            f'  {processed_total:,}/{total:,} ({processed_total/total*100:.1f}%) | '
            f'avg {avg_time:.1f}s/tweet | ETA ~{eta_sec/60:.1f} min'
        )

    if n % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(checkpoint_path, index=False)
        print(f'  [Checkpoint saved: {len(results):,} rows]')

elapsed    = time.time() - start_time
df_results = pd.DataFrame(results)
null_count = df_results['pred_label'].isnull().sum()

df_results.to_csv(raw_path, index=False)
if checkpoint_path.exists():
    checkpoint_path.unlink()

print(f'\nDone! {len(df_results):,} classified in {elapsed/60:.1f} min')
print(f'Null / parse failures: {null_count} ({null_count/len(df_results)*100:.1f}%)')
print(f'\nParse method breakdown:')
for method, count in df_results['parse_method'].value_counts().items():
    print(f'  {method:<22}: {count:,} ({count/len(df_results)*100:.1f}%)')

## 7. Handle Errors & Finalize Predictions

In [ ]:
null_mask = df_results['pred_label'].isnull()
print(f'Null predictions: {null_mask.sum()} / {len(df_results)} ({null_mask.sum()/len(df_results)*100:.1f}%)')

if null_mask.sum() > 0:
    print('\nSample failed predictions:')
    print(df_results[null_mask][['text', 'reasoning', 'parse_method']].head(3).to_string())

majority_class = df_results['true_label'].mode()[0]
df_results['pred_label_final'] = df_results['pred_label'].fillna(majority_class)
df_results.loc[null_mask, 'confidence'] = df_results.loc[null_mask, 'confidence'].replace(0.0, 0.5)

print(f'\nFilled {null_mask.sum()} nulls with majority class: "{majority_class}"')
print(f'\nPrediction distribution:')
print(df_results['pred_label_final'].value_counts())

## 8. Evaluation Metrics

In [ ]:
y_true = df_results['true_label'].map(LABEL_MAP).values
y_pred = df_results['pred_label_final'].map(LABEL_MAP).values

print(f'\n{"="*60}')
print(f' {DATASET.upper()} — {MODEL_CFG["display_name"]} Zero-Shot Results')
print(f'{"="*60}')
print(classification_report(y_true, y_pred, target_names=CFG['label_names'], digits=4))

pos_label_int = LABEL_MAP[CFG['pos_label']]
parse_counts  = df_results['parse_method'].value_counts().to_dict()

metrics = {
    'dataset':              DATASET,
    'model':                MODEL,
    'model_display':        MODEL_CFG['display_name'],
    'params_b':             MODEL_CFG['params_b'],
    'openrouter_id':        MODEL_CFG['openrouter_id'],
    # test_ prefix for compatibility with notebook 09
    'test_accuracy':        accuracy_score(y_true, y_pred),
    'test_f1_macro':        f1_score(y_true, y_pred, average='macro'),
    'test_f1_weighted':     f1_score(y_true, y_pred, average='weighted'),
    'test_precision':       precision_score(y_true, y_pred, average='macro', zero_division=0),
    'test_recall':          recall_score(y_true, y_pred, average='macro', zero_division=0),
    f'test_f1_{CFG["pos_label"]}': f1_score(y_true, y_pred, pos_label=pos_label_int, zero_division=0),
    'null_predictions':     int(null_mask.sum()),
    'parse_errors':         int(df_results['parse_error'].sum()),
    'parse_json':           parse_counts.get('json', 0),
    'parse_keyword_fallback': parse_counts.get('keyword_fallback', 0),
    'parse_empty':          parse_counts.get('empty', 0),
    'parse_default':        parse_counts.get('default', 0),
}

print('--- Summary ---')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<40}: {v:.4f}')
    else:
        print(f'  {k:<40}: {v}')

## 9. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['label_names'], yticklabels=CFG['label_names'], ax=axes[0])
axes[0].set_title(f'{DATASET.upper()} — {MODEL_CFG["display_name"]} (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CFG['label_names'], yticklabels=CFG['label_names'], ax=axes[1])
axes[1].set_title(f'{DATASET.upper()} — {MODEL_CFG["display_name"]} (Normalized)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_{MODEL}_cm.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## 10. Confidence Distribution

In [ ]:
df_results['correct'] = (df_results['true_label'] == df_results['pred_label_final'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

correct_conf   = df_results[df_results['correct']]['confidence']
incorrect_conf = df_results[~df_results['correct']]['confidence']

axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label=f'Correct (n={len(correct_conf)})')
axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='crimson',   label=f'Incorrect (n={len(incorrect_conf)})')
axes[0].set_xlabel('Confidence'); axes[0].set_ylabel('Count')
axes[0].set_title(f'{DATASET.upper()} — Confidence by Outcome', fontweight='bold')
axes[0].legend(); axes[0].axvline(0.5, color='black', linestyle='--', alpha=0.5)

for label_name in CFG['label_names']:
    subset = df_results[df_results['pred_label_final'] == label_name]['confidence']
    axes[1].hist(subset, bins=20, alpha=0.6, label=f'{label_name} (n={len(subset)})')
axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count')
axes[1].set_title(f'{DATASET.upper()} — Confidence by Predicted Label', fontweight='bold')
axes[1].legend()

plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_{MODEL}_confidence.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean confidence — correct: {correct_conf.mean():.3f} | incorrect: {incorrect_conf.mean():.3f}')

## 11. Save Predictions & Summary

In [ ]:
df_results['true_label_int'] = df_results['true_label'].map(LABEL_MAP)
df_results['pred_label_int'] = df_results['pred_label_final'].map(LABEL_MAP)

pred_path    = PREDS_DIR / f'{DATASET}_{MODEL}_test_predictions.csv'
summary_path = PREDS_DIR / f'{DATASET}_{MODEL}_summary.csv'

df_results.to_csv(pred_path, index=False)
pd.DataFrame([metrics]).to_csv(summary_path, index=False)

print(f'Predictions saved : {pred_path}')
print(f'Summary saved     : {summary_path}')
print(f'\nColumns: {list(df_results.columns)}')

print(f'\n=== FINAL RESULTS ===')
print(f'  Dataset  : {DATASET.upper()}')
print(f'  Model    : {MODEL_CFG["display_name"]}')
print(f'  Accuracy : {metrics["test_accuracy"]:.4f}')
print(f'  F1 Macro : {metrics["test_f1_macro"]:.4f}')
print(f'  Precision: {metrics["test_precision"]:.4f}')
print(f'  Recall   : {metrics["test_recall"]:.4f}')

## 12. Results Summary Table

In [ ]:
pos_key = f'test_f1_{CFG["pos_label"]}'

metrics_display = [
    ['Metric',                  'Value'],
    ['Accuracy',                f"{metrics['test_accuracy']:.4f}"],
    ['F1 Macro',                f"{metrics['test_f1_macro']:.4f}"],
    ['F1 Weighted',             f"{metrics['test_f1_weighted']:.4f}"],
    ['Precision (Macro)',       f"{metrics['test_precision']:.4f}"],
    ['Recall (Macro)',          f"{metrics['test_recall']:.4f}"],
    [f'F1 ({CFG["pos_label"]})',f"{metrics[pos_key]:.4f}"],
    ['Parse: JSON',             str(metrics['parse_json'])],
    ['Parse: keyword fallback', str(metrics['parse_keyword_fallback'])],
    ['Parse: empty/default',    str(metrics['parse_empty'] + metrics['parse_default'])],
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')
table = ax.table(
    cellText=metrics_display[1:],
    colLabels=metrics_display[0],
    cellLoc='center', loc='center',
    bbox=[0.15, 0, 0.7, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2980b9')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#eaf4fb')

ax.set_title(
    f'{DATASET.upper()} — {MODEL_CFG["display_name"]} Zero-Shot Results',
    fontsize=13, fontweight='bold', pad=20
)
plt.tight_layout()
fig_path = FIGS_DIR / f'{DATASET}_{MODEL}_summary_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')
print(f'\nNext: change DATASET / MODEL at the top and re-run all cells.')